# 🖥️ Which RunPod GPU should I use?

This notebook loads **openai/gpt-oss-20b** (~21.5 B params, Mixture-of-Experts, ~3.6 B active/token). It ships **natively in MXFP4 4-bit**, but MXFP4's compute kernels only run on **Hopper (H100/H200)** — on older cards transformers **dequantizes to bf16**. So the VRAM you need depends on the GPU generation:

| Precision | Weights | ~VRAM (peak gen) | Runs on | Recommended GPU | ~Price/hr (USD) |
|:---|:---|:---|:---|:---|:---|
| **MXFP4** (native 4-bit) | ~13 GB | ~15 GB | **Hopper only** (H100/H200) | **H100 · 80 GB** | ~2.0–2.8 |
| **bf16** (dequantized) | ~42 GB | ~46 GB | any ≥48 GB card | **A40 or A6000 · 48 GB** | ~0.40–0.49 |
| **fp32** (full) | ~86 GB | ~90 GB | multi-GPU only | 2× 80 GB | — |

_Prices: RunPod Community Cloud, on-demand, ~July 2026 — check [runpod.io/pricing](https://www.runpod.io/pricing). Set the pod's **Container/Volume disk to ≥ 100 GB** (the download is ~28 GB)._

> **On this A6000 (48 GiB ≈ 51.5 GB decimal): use `bf16`** — verified at ~46 GB peak VRAM and ~29 tok/s. MXFP4 silently falls back to bf16 here (no Hopper), and fp32 (~86 GB) will not fit on one card.

# Install libraries

In [1]:
%pip install torch
%pip install --upgrade transformers
%pip install accelerate
%pip install peft
%pip install datasets
%pip install evaluate
%pip install trl
%pip install matplotlib
%pip install tensorboard
%pip install sentencepiece

# Optional -- ONLY on Hopper (H100/H200) to keep gpt-oss in native MXFP4 (~15 GB VRAM).
# On Ampere/Ada it has no effect (the model still dequantizes to bf16).
# %pip install --no-deps "kernels>=0.12.0"

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i).name)

NVIDIA RTX A6000


In [3]:
from huggingface_hub import login

API_TOKEN = "HF_TOKEN_REMOVED"
login(token=API_TOKEN)

# ⚙️ Choose precision -- run ONE of the three cells below

Run **exactly one** of the MXFP4 / bf16 / fp32 cells. It sets `PRECISION` and `MODEL_KWARGS`,
used by every section below (loading, the on-disk folder `SAVED_MODEL_GPTOSS20B_<precision>`, and the
Hub repo names `my-base-gpt-oss-20b-<precision>-...`). To switch precision later, run a different cell,
then re-run the **Load the model** cell.

**On this A6000, use `bf16`.** MXFP4 needs a Hopper GPU (otherwise it falls back to bf16); fp32 (~86 GB) will not fit on one card.

In [ ]:
# --- MXFP4 (native 4-bit) -- Hopper (H100/H200) only; needs `pip install kernels` ---
# On non-Hopper GPUs transformers silently dequantizes this to bf16 (~42 GB).
import torch

PRECISION = "mxfp4"
MODEL_KWARGS = dict(
    device_map="cuda",   # no dtype -> keep the checkpoint's native MXFP4 if kernels + Hopper are present
)
print("Selected precision:", PRECISION)

In [4]:
# --- bf16 (dequantized half precision) -- works on this A6000 ---
import torch

PRECISION = "bf16"
MODEL_KWARGS = dict(
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)

Selected precision: bf16


In [ ]:
# --- fp32 (full precision) -- ~86 GB, needs multi-GPU; will NOT fit on a single 48 GB card ---
import torch

PRECISION = "fp32"
MODEL_KWARGS = dict(
    dtype=torch.float32,
    device_map="auto",
)
print("Selected precision:", PRECISION)

# Base model from HUB

**What this notebook does:** download the original `openai/gpt-oss-20b` from the Hugging Face Hub at the precision you selected, verify it generates a coherent answer, then persist it — save a copy to local disk **and** push a copy to your own Hub repo.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>gpt-oss-20b"] -->|load| BM["base_model"]
    BM -->|test| OK["✅ generates"]
    BM -->|save| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|push| HUB["📦 Your Hub<br/>…-from-hub"]
    style ORIG fill:#2e7d32,color:#fff,stroke:#1b5e20
```

### Load the model

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Silence transformers' verbose warnings.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

model_name = "openai/gpt-oss-20b"

# Names derived from the precision you selected above -- used by every section below.
SAVED_DIR      = f"SAVED_MODEL_GPTOSS20B_{PRECISION}"
REPO_FROM_HUB  = f"lukaskellerstein/my-base-gpt-oss-20b-{PRECISION}-from-hub"
REPO_FROM_DISK = f"lukaskellerstein/my-base-gpt-oss-20b-{PRECISION}-from-disk"

base_model = AutoModelForCausalLM.from_pretrained(model_name, **MODEL_KWARGS)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loaded {model_name}  |  precision={PRECISION}  |  dtype={next(base_model.parameters()).dtype}")
print(f"hub repo={REPO_FROM_HUB}")

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

Loaded openai/gpt-oss-20b  |  precision=bf16  |  dtype=torch.bfloat16
hub repo=lukaskellerstein/my-base-gpt-oss-20b-bf16-from-hub


Log model and tokenizer

In [7]:
def describe(model, tokenizer):
    """Print key facts about a loaded model + tokenizer. Reused in every section below."""
    print("--- Model ---")
    print("Type:", type(model))
    print("Architecture:", model)
    # get_text_config() returns the text sub-config for multimodal models and the
    # config itself for plain text models (like gpt-oss), so this works everywhere.
    print("Vocab size:", model.config.get_text_config().vocab_size)
    print("Input embeddings:", model.get_input_embeddings())
    print("Output embeddings:", model.get_output_embeddings())
    print("--- Tokenizer ---")
    print("Type:", type(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)
    print("All tokens count:", len(tokenizer))
    print("Padding side:", tokenizer.padding_side)


describe(base_model, base_tokenizer)

--- Model ---
Type: <class 'transformers.models.gpt_oss.modeling_gpt_oss.GptOssForCausalLM'>
Architecture: GptOssForCausalLM(
  (model): GptOssModel(
    (embed_tokens): Embedding(201088, 2880, padding_idx=199999)
    (layers): ModuleList(
      (0-23): 24 x GptOssDecoderLayer(
        (self_attn): GptOssAttention(
          (q_proj): Linear(in_features=2880, out_features=4096, bias=True)
          (k_proj): Linear(in_features=2880, out_features=512, bias=True)
          (v_proj): Linear(in_features=2880, out_features=512, bias=True)
          (o_proj): Linear(in_features=4096, out_features=2880, bias=True)
        )
        (mlp): GptOssMLP(
          (router): GptOssTopKRouter()
          (experts): GptOssExperts()
        )
        (input_layernorm): GptOssRMSNorm((2880,), eps=1e-05)
        (post_attention_layernorm): GptOssRMSNorm((2880,), eps=1e-05)
      )
    )
    (norm): GptOssRMSNorm((2880,), eps=1e-05)
    (rotary_emb): GptOssRotaryEmbedding()
  )
  (lm_head): Linear(in_fea

### Test the model

#### via Pipeline

In [8]:
from transformers import pipeline

PROMPT = "What is AutoGen in abstract?"


def _final(text):
    """gpt-oss uses OpenAI's 'harmony' format: a chain-of-thought 'analysis' channel
    followed by the answer after 'assistantfinal'. Return just the final answer."""
    return text.split("assistantfinal")[-1].strip() if "assistantfinal" in text else text


def test_pipeline(model, tokenizer, prompt=PROMPT):
    """Test a model with a text-generation pipeline. Reused in every section below."""
    my_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # Instruction-tuned model -> pass a chat (list of messages); the pipeline applies
    # the chat template. gpt-oss reasons before answering, so allow enough tokens.
    messages = [{"role": "user", "content": prompt}]
    result = my_pipeline(messages, max_new_tokens=512, return_full_text=False)
    gen = result[0]["generated_text"]
    if isinstance(gen, list):          # some versions return the message list
        gen = gen[-1]["content"]
    print(_final(gen))


test_pipeline(base_model, base_tokenizer)

analysisThe user: "What is AutoGen in abstract?" They likely refer to a concept named AutoGen. Could be AutoGen from OpenAI? AutoGen is a framework for building autonomous AI agents? In abstract, it's a framework that orchestrates multiple LLMs with self-reflection and self-correction. The question likely wants a concise abstract definition: "AutoGen is an AI framework for building autonomous agents that can generate content, reason, and interact in dialogues." But we need to be careful: "AutoGen" might refer to "AutoGen: A framework for building large language model powered applications," or "AutoGen" from "AutoGen: A framework for building AI agents," or "AutoGen" from "AutoGen: A framework for generating code." Let's research memory: There's an AutoGen by Microsoft? Actually, AutoGen is a project by "Microsoft" maybe? No, AutoGen is a project by "OpenAI" or "Google"? Wait: AutoGen: "AutoGen: A Framework for Building Autonomous Agents with LLMs" by OpenAI? I recall a paper "AutoGen: 

#### via Model and Tokenizer

In [9]:
def test_generate(model, tokenizer, prompt=PROMPT):
    """Test a model by manually applying the chat template + generate. Reused below."""
    messages = [{"role": "user", "content": prompt}]
    # gpt-oss reasons before answering; reasoning_effort="low" keeps the hidden
    # 'analysis' channel short so the final answer fits comfortably in max_new_tokens.
    model_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True,
        add_generation_prompt=True, reasoning_effort="low",
    ).to(model.device)

    result = model.generate(**model_inputs, max_new_tokens=512)
    text = tokenizer.decode(
        result[0][model_inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(_final(text))  # keep only the answer after the 'assistantfinal' marker


test_generate(base_model, base_tokenizer)

**AutoGen in a nutshell**

AutoGen is an *autonomous code‑generation framework* (short for “automatic generation”) that turns a single‑purpose language model (LLM) into a *multi‑agent ecosystem* that can:

1. **Self‑directed exploration** – let the agent pose questions to itself, gather data, and evaluate its own performance.  
2. **Modular composition** – plug in any number of “helper” agents (e.g., a summarizer, a code validator, a text‑to‑text translator) that communicate through a simple prompt‑based interface.  
3. **Feedback‑driven loop** – receive feedback, refine its plan, and retry until a specification is met.

The core idea is that instead of writing a long monolithic prompt, you give the LLM a *workflow* and a *set of responsibilities* and let it orchestrate the rest. This makes LLMs more reliable for complex, iterative tasks such as software synthesis, research drafting, or problem solving with external APIs.


#### via Tool call (single turn)

In [10]:
# Single-turn tool calling: pass `tools` to the chat template. An instruction-tuned
# model should answer with a tool call (function name + JSON arguments) instead of prose.
STOCK_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current price of a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The ticker symbol for the stock, e.g. GOOG",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]


def test_tool_call(model, tokenizer, prompt, tools, max_new_tokens=256):
    """Single-turn tool calling: pass `tools`; the model should emit a tool call. Reused below."""
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Greedy decoding -> deterministic tool call. Keep special tokens so the tool-call
    # markers (gpt-oss emits them in its 'commentary' channel) stay visible.
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_call(base_model, base_tokenizer, "What is the current price of MSFT?", STOCK_TOOLS)


<|channel|>analysis<|message|>We need to call the function get_stock_price with ticker "MSFT".<|end|><|start|>assistant<|channel|>commentary to=functions.get_stock_price <|constrain|>json<|message|>{"ticker":"MSFT"}<|call|>


#### via Conversation with tool calls

In [14]:
def test_tool_conversation(model, tokenizer, max_new_tokens=256):
    """Multi-turn conversation that already contains an assistant tool call + a tool
    response, then a fresh user turn. Confirms the chat template round-trips the
    `tool_calls` / `tool` roles, then answers the latest turn. Reused below."""
    conversation = [
        {"role": "user", "content": "What's the weather like in Paris?"},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call12345",
                    "type": "function",
                    "function": {
                        "name": "get_current_weather",
                        "arguments": '{"location": "Paris, France", "format": "celsius"}',
                    },
                }
            ],
        },
        {"role": "tool", "content": '{"content": 22}', "tool_call_id": "call12345"},
        {
            "role": "assistant",
            "content": "The current temperature in Paris, France is 22 degrees Celsius.",
        },
        {"role": "user", "content": "What's the weather like in San Francisco right now?"},
    ]

    weather_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "format": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use. Infer this from the users location.",
                        },
                    },
                    "required": ["location", "format"],
                },
            },
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        conversation, tools=weather_tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    print("Tokenized conversation input_ids:")
    print(model_inputs["input_ids"])

    # Answer the latest user turn (should be another get_current_weather tool call for SF).
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print("\nModel response:")
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_conversation(base_model, base_tokenizer)


Tokenized conversation input_ids:
tensor([[200006,  17360, 200008,   3575,    553,  17554, 162016,     11,    261,
           4410,   6439,   2359,  22203,    656,   7788,  17527,    558,  87447,
         100594,     25,    220,   1323,     19,     12,   3218,    198,   6576,
           3521,     25,    220,   1323,     21,     12,   3173,     12,   3218,
            279,  30377,    289,     25,  14093,    279,      2,  13888,  18403,
             25,   8450,     11,  49159,     11,   1721,     13,  21030,   2804,
            413,   7360,    395,   1753,   3176,    558,  63446,    316,   1879,
           8437,   2804,    810,    316,    290,  49159,   9334,     25,    461,
          44580,   6120, 200007, 200006,  77944, 200008,      2,  20574,    279,
            877,   9964,    279,   4797,   9964,  95359,   2942,    290,   2208,
          11122,    198,   2493,    717,  23981, 170154,    314,  11350,     25,
          10168,    623,   5030,    326,   2608,     11,    319,   1940,   


Model response:
<|channel|>analysis<|message|>We need to call get_current_weather with location "San Francisco, CA" and format "fahrenheit" (since user is in US).<|end|><|start|>assistant<|channel|>commentary to=functions.get_current_weather<|constrain|>json<|message|>{"location":"San Francisco, CA","format":"fahrenheit"}<|call|>


### Save the model (to disk)

In [12]:
base_model.save_pretrained(SAVED_DIR)

base_tokenizer.save_pretrained(SAVED_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('SAVED_MODEL_GPTOSS20B_bf16/tokenizer_config.json',
 'SAVED_MODEL_GPTOSS20B_bf16/chat_template.jinja',
 'SAVED_MODEL_GPTOSS20B_bf16/tokenizer.json')

### Push the model (to HUB)

In [ ]:
# push_to_hub() -> save_pretrained(save_original_format=True) can raise NotImplementedError
# in transformers 5.x, so save with save_original_format=False and upload the folder instead.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{PRECISION}_from_hub"
create_repo(REPO_FROM_HUB, token=API_TOKEN, exist_ok=True)
base_model.save_pretrained(tmp, save_original_format=False)
base_tokenizer.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_HUB, folder_path=tmp, token=API_TOKEN)